##Data Reading

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@saolympicsproj.dfs.core.windows.net/athletes")

In [ ]:
display(df)

In [ ]:
df = df.fillna({"birth_place":"xyz", "birth_country":"abc", "residence_place":"unknown", "residence_country":"aaa"})

In [ ]:
df.display()

In [ ]:
df_filtered = df.filter((col('current')==True) & (col('name').isin('GALSTYAN Slavik','HARUTYUNYAN Arsen','SEHEN Sajjad')))

In [ ]:
df_filtered.display()

In [ ]:
df = df.withColumn('height',col('height').cast(FloatType()))\
    .withColumn('weight',col('weight').cast(FloatType()))

In [ ]:
df.display()

In [ ]:
df_sorted = df.sort('height','weight',ascending=[0,1]).filter(col('weight')>0)
df_sorted.display()

In [ ]:
df_sorted =  df_sorted.withColumn('nationality',regexp_replace('nationality', 'United States', 'US'))
df_sorted.display()

In [ ]:
df.groupBy('code').agg(count('code').alias('total_count')).filter(col('total_count')>1).display()

In [ ]:
df_sorted = df_sorted.withColumnRenamed("code","athelete_id")
df_sorted.display()

In [ ]:
df_sorted = df_sorted.withColumn('occupation',split('occupation',','))

In [ ]:
df_sorted.display()

In [ ]:
df_sorted.columns

In [ ]:
df_final = df_sorted.select('athelete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

In [ ]:
display(df_final)

In [ ]:
from pyspark.sql.window import Window

In [ ]:
df_final.withColumn("cum_weight", sum("weight").over(Window.partitionBy("nationality").orderBy("height").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

In [ ]:
df_final.createOrReplaceTempView("athletes")

In [ ]:
df_new = spark.sql("""
            select sum(weight) over(partition by nationality order by height rows between unbounded preceding and current row)
            from athletes
          """)

In [ ]:
df_new.display()

In [ ]:
df_final.write.format("delta")\
    .mode("append")\
    .option("path","abfss://silver@saolympicsproj.dfs.core.windows.net/athletes")\
    .saveAsTable("olympics.silver.athletes")